In [4]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, f1_score, recall_score, precision_score
from sklearn.metrics import precision_recall_curve

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    precision_recall_curve,
    confusion_matrix,
    classification_report,
    average_precision_score
)
from xgboost import XGBClassifier

# =========================
# 1️⃣ Chargement des données
# =========================

train = pd.read_csv('../6.Data/Yann_Process_train.csv')
test = pd.read_csv('../6.Data/Yann_Process_test.csv')

train = train.drop(columns="account_age_days")
test = test.drop(columns="account_age_days")

target = 'target_is_fraud'
id_col = 'customer_id'

selected_features = [col for col in train.columns if col not in [target, id_col]]


def feature_engineering(df):
    df = df.copy()

    # ── Bloc 2 : risque technique device/IP ─────────────────────────
    df['device_ip_risk']   = df['ip_risk_z'] - df['device_trust_z']   # TOP feature (p=1e-250)


    # ── Bloc 3 : historique de problèmes ────────────────────────────
    df['trouble_score']    = (df['chargebacks_12m']
                              + df['failed_payments_6m']
                              + df['support_tickets_90d'])


    return df


train = feature_engineering(train)
test  = feature_engineering(test)

X = train[selected_features]
y = train[target]
X_test = test[selected_features]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# =========================
# 2️⃣ Gestion déséquilibre
# =========================

scale = (y_train == 0).sum() / (y_train == 1).sum()

model = XGBClassifier(
    scale_pos_weight=scale,
    random_state=42,
    n_jobs=-1,
    eval_metric='aucpr'
)

# =========================
# 3️⃣ GridSearch
# =========================

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 7, 10],
    'learning_rate': [0.01, 0.1],
}

grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring='average_precision',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("Best params:", grid.best_params_)

# =========================
# 4️⃣ Recherche du meilleur seuil (max F1-score)
# =========================

y_proba = best_model.predict_proba(X_val)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)

# Calcul du F1-score pour chaque seuil
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-9)

# Indice du meilleur F1
best_idx = np.argmax(f1_scores)

best_threshold = thresholds[best_idx]

print(f"Meilleur seuil (F1 max) : {best_threshold:.3f}")
print(f"Meilleur F1-score : {f1_scores[best_idx]:.4f}")

# =========================
# 5️⃣ Évaluation finale
# =========================

y_pred = (y_proba >= best_threshold).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

print("\nClassification Report:")
print(classification_report(y_val, y_pred))

print("\nAverage Precision Score:")
print(average_precision_score(y_val, y_proba))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}
Meilleur seuil (F1 max) : 0.720
Meilleur F1-score : 0.4078

Confusion Matrix:
[[26772  2473]
 [ 1416  1339]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.92      0.93     29245
           1       0.35      0.49      0.41      2755

    accuracy                           0.88     32000
   macro avg       0.65      0.70      0.67     32000
weighted avg       0.90      0.88      0.89     32000


Average Precision Score:
0.3862934617693901
